<a href="https://colab.research.google.com/github/ChNavya-777/Mobile_Addiction_Level_Predictor/blob/main/Mobile_Addiction_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("teen_phone_addiction_dataset.csv")

print(data.head())
print(data.info())

   ID               Name  Age  Gender          Location School_Grade  \
0   1    Shannon Francis   13  Female        Hansonfort          9th   
1   2    Scott Rodriguez   17  Female      Theodorefort          7th   
2   3        Adrian Knox   13   Other       Lindseystad         11th   
3   4  Brittany Hamilton   18  Female      West Anthony         12th   
4   5       Steven Smith   14   Other  Port Lindsaystad          9th   

   Daily_Usage_Hours  Sleep_Hours  Academic_Performance  Social_Interactions  \
0                4.0          6.1                    78                    5   
1                5.5          6.5                    70                    5   
2                5.8          5.5                    93                    8   
3                3.1          3.9                    78                    8   
4                2.5          6.7                    56                    4   

   ...  Screen_Time_Before_Bed  Phone_Checks_Per_Day  Apps_Used_Daily  \
0  ...       

In [2]:
data = data.drop(["ID", "Name", "Location"], axis=1)

In [3]:
print(data.isnull().sum())

# Fill numeric with median
for col in data.select_dtypes(include=np.number).columns:
    data[col].fillna(data[col].median(), inplace=True)

# Fill categorical with mode
for col in data.select_dtypes(include="object").columns:
    data[col].fillna(data[col].mode()[0], inplace=True)

Age                       0
Gender                    0
School_Grade              0
Daily_Usage_Hours         0
Sleep_Hours               0
Academic_Performance      0
Social_Interactions       0
Exercise_Hours            0
Anxiety_Level             0
Depression_Level          0
Self_Esteem               0
Parental_Control          0
Screen_Time_Before_Bed    0
Phone_Checks_Per_Day      0
Apps_Used_Daily           0
Time_on_Social_Media      0
Time_on_Gaming            0
Time_on_Education         0
Phone_Usage_Purpose       0
Family_Communication      0
Weekend_Usage_Hours       0
Addiction_Level           0
dtype: int64


/tmp/ipykernel_4743/2388132496.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data[col].fillna(data[col].median(), inplace=True)
/tmp/ipykernel_4743/2388132496.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', tr

In [4]:
from sklearn.preprocessing import LabelEncoder

encoders = {}
categorical_cols = ["Gender", "School_Grade", "Phone_Usage_Purpose"]

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

In [5]:
Q1 = data.quantile(0.25)
Q3 = data.quantile(0.75)
IQR = Q3 - Q1

data = data[~((data < (Q1 - 1.5 * IQR)) |
              (data > (Q3 + 1.5 * IQR))).any(axis=1)]

print("After removing outliers:", data.shape)

After removing outliers: (2857, 22)


In [6]:
X = data.drop("Addiction_Level", axis=1)
y = data["Addiction_Level"]

In [7]:
from sklearn.feature_selection import SelectKBest, f_regression

selector = SelectKBest(score_func=f_regression, k=10)
X_selected = selector.fit_transform(X, y)

selected_columns = X.columns[selector.get_support()]
print("Selected Features:", selected_columns)

# Convert back to DataFrame
X = pd.DataFrame(X_selected, columns=selected_columns)

Selected Features: Index(['Age', 'Daily_Usage_Hours', 'Sleep_Hours', 'Social_Interactions',
       'Exercise_Hours', 'Self_Esteem', 'Phone_Checks_Per_Day',
       'Apps_Used_Daily', 'Time_on_Social_Media', 'Time_on_Gaming'],
      dtype='object')


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=selected_columns)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=100, random_state=42)
dt = DecisionTreeRegressor(random_state=42)

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
dt.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_pred_rf = rf.predict(X_test)
y_pred_dt = dt.predict(X_test)

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(y_test, y_pred, name):
    print(f"\n{name}")
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
    print("R2:", r2_score(y_test, y_pred))

evaluate(y_test, y_pred_lr, "Linear Regression")
evaluate(y_test, y_pred_rf, "Random Forest")
evaluate(y_test, y_pred_dt, "Decision Tree")


Linear Regression
MAE: 0.6037067410167
RMSE: 0.7350646990770772
R2: 0.7199328680335184

Random Forest
MAE: 0.34903671328671343
RMSE: 0.5112684948011406
R2: 0.8645094286761281

Decision Tree
MAE: 0.49353146853146856
RMSE: 0.8134468662400786
R2: 0.6570195296118663


In [12]:
best_model = rf  # Usually best for this dataset
print("Best Model:", best_model)

Best Model: RandomForestRegressor(random_state=42)


In [13]:
import pickle

with open("phone_addiction_model.pkl", "wb") as f:
    pickle.dump({
        "model": best_model,
        "scaler": scaler,
        "encoders": encoders,
        "features": list(selected_columns)
    }, f)

In [14]:
import gradio as gr
import pandas as pd
import pickle

# ===============================
# LOAD SAVED MODEL
# ===============================
with open("phone_addiction_model.pkl", "rb") as f:
    saved_data = pickle.load(f)

model = saved_data["model"]
scaler = saved_data["scaler"]
encoders = saved_data["encoders"]
feature_order = saved_data["features"]

# ===============================
# FILTER ENCODERS (IMPORTANT)
# ===============================
filtered_encoders = {
    col: encoders[col] for col in encoders if col in feature_order
}

# ===============================
# PREDICTION FUNCTION
# ===============================
def predict_addiction(*inputs):
    try:
        # Convert to DataFrame
        input_df = pd.DataFrame([inputs], columns=feature_order)

        # Encode categorical columns
        for col in filtered_encoders:
            input_df[col] = filtered_encoders[col].transform(input_df[col])

        # Maintain correct order
        input_df = input_df[feature_order]

        # Scale numerical columns
        num_cols = [col for col in feature_order if col not in filtered_encoders]
        input_df[num_cols] = scaler.transform(input_df[num_cols])

        # Prediction
        prediction = model.predict(input_df)[0]

        # Convert to level
        if prediction < 4:
            level = "Low 🟢"
            suggestion = "Good usage. Maintain healthy habits."
        elif prediction < 7:
            level = "Moderate 🟡"
            suggestion = "Try reducing screen time, especially before sleep."
        else:
            level = "High 🔴"
            suggestion = "High usage detected. Reduce screen time and take breaks."

        return f"📊 Score: {prediction:.2f}\n🎯 Level: {level}\n💡 Suggestion: {suggestion}"

    except Exception as e:
        return f"Error: {str(e)}"


# ===============================
# UI INPUTS (AUTO-GENERATED)
# ===============================
ui_inputs = []

for col in feature_order:
    if col in filtered_encoders:
        ui_inputs.append(
            gr.Dropdown(
                choices=list(filtered_encoders[col].classes_),
                label=col
            )
        )
    else:
        ui_inputs.append(
            gr.Number(label=col)
        )

# ===============================
# INTERFACE
# ===============================
iface = gr.Interface(
    fn=predict_addiction,
    inputs=ui_inputs,
    outputs=gr.Textbox(label="Prediction Result"),
    title="📱 Mobile Addiction Level Predictor",
    description="""
Enter user details to predict addiction score.

Levels:
🟢 Low (<4)
🟡 Moderate (4–7)
🔴 High (>7)
"""
)

# ===============================
# LAUNCH
# ===============================
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://13285a24bac9ef2f37.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
